# ATIV-04-ET-02 - Segmentação de Tumor Cerebral
## Germ Cell Tumor (Germinoma)

**Objetivo:** Apresentar métodos da literatura e propor ideias para métodos próprios.

**Dataset:** Brain Tumor 12K MRI Images w Masks, Meta and Bbox (Kaggle)

**Tumor:** Germ Cell Tumor (Germinoma) - 263 imagens

## 1. Setup e Download dos Dados

In [ ]:
!pip install -q segmentation-models-pytorch albumentations kaggle

In [ ]:
import os
import json
import glob
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# Verificar GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Configurar Kaggle e baixar dataset
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Username Kaggle: ')
os.environ['KAGGLE_KEY'] = getpass('API Key: ')

!kaggle datasets download -d fernando2rad/brain-tumor-12k-mri-images-w-masks-meta-and-bbox \
    -p /content/dataset --force --quiet

!unzip -qo /content/dataset/*.zip -d /content/dataset
print('Download concluido!')

In [ ]:
# Ver a estrutura do dataset
for root, dirs, files in os.walk('/content/dataset'):
    level = root.replace('/content/dataset', '').count(os.sep)
    if level <= 2:
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/ ({len(files)} arquivos)')

## 2. Carregar e Filtrar os Dados do Germinoma

In [ ]:
# Encontrar o JSON de metadados
json_path = glob.glob('/content/dataset/**/DATA.json', recursive=True)[0]
dataset_root = os.path.dirname(json_path)
print(f'JSON: {json_path}')
print(f'Raiz: {dataset_root}')

with open(json_path, 'r', encoding='utf-8') as f:
    all_data = json.load(f)

print(f'Total de entradas: {len(all_data)}')

# Ver estrutura de uma entrada
if isinstance(all_data, list):
    print('\nExemplo de entrada:')
    for k, v in all_data[0].items():
        print(f'  {k}: {v}')

In [ ]:
# Filtrar somente as entradas de Germinoma / Germ Cell
if isinstance(all_data, list):
    df_all = pd.DataFrame(all_data)
else:
    records = []
    for key, val in all_data.items():
        if isinstance(val, dict):
            val['id'] = key
            records.append(val)
    df_all = pd.DataFrame(records)

print(f'Colunas: {list(df_all.columns)}')
print(f'Shape: {df_all.shape}')

# Buscar registros de Germinoma em qualquer coluna de texto
mask_germ = pd.Series([False] * len(df_all))
for col in df_all.columns:
    if df_all[col].dtype == 'object':
        mask_germ = mask_germ | df_all[col].str.contains(
            'Germinoma|Germ Cell', case=False, na=False
        )

df_germ = df_all[mask_germ].copy().reset_index(drop=True)
print(f'\nImagens de Germinoma encontradas: {len(df_germ)}')
df_germ.head()

In [ ]:
# Distribuicao por classe
for col in df_germ.columns:
    if df_germ[col].dtype == 'object' and 1 < df_germ[col].nunique() <= 10:
        print(f'\n{col}:')
        print(df_germ[col].value_counts())

In [ ]:
# Identificar colunas de caminho da imagem e da mascara
def find_file(filename, search_root='/content/dataset'):
    for base in [search_root, dataset_root]:
        full = os.path.join(base, filename)
        if os.path.exists(full):
            return full
    results = glob.glob(f'{search_root}/**/{os.path.basename(filename)}', recursive=True)
    return results[0] if results else None

# Descobrir qual coluna tem o caminho da imagem e da mascara
img_col = None
mask_col = None

for col in df_germ.columns:
    cl = col.lower()
    sample_val = str(df_germ[col].iloc[0]).lower()
    if 'mask' in cl and ('path' in cl or any(e in sample_val for e in ['.png','.jpg'])):
        mask_col = col
    elif ('file' in cl or 'image' in cl or 'path' == cl) and any(e in sample_val for e in ['.png','.jpg','.jpeg']):
        img_col = col

# Fallback
if img_col is None:
    for col in df_germ.columns:
        val = str(df_germ[col].iloc[0])
        if any(ext in val.lower() for ext in ['.png','.jpg']) and 'mask' not in col.lower():
            img_col = col
            break
if mask_col is None:
    for col in df_germ.columns:
        if 'mask' in col.lower():
            mask_col = col
            break

print(f'Coluna imagem: {img_col} -> ex: {df_germ[img_col].iloc[0]}')
print(f'Coluna mascara: {mask_col} -> ex: {df_germ[mask_col].iloc[0]}')

# Testar
print(f'\nImagem encontrada: {find_file(str(df_germ[img_col].iloc[0]))}')
print(f'Mascara encontrada: {find_file(str(df_germ[mask_col].iloc[0]))}')

In [ ]:
# Visualizar algumas amostras com suas mascaras
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
samples = df_germ.sample(n=min(6, len(df_germ)), random_state=42)

for i, (_, entry) in enumerate(samples.iterrows()):
    row = i // 2
    col_start = (i % 2) * 2

    img_path = find_file(str(entry[img_col]))
    if img_path:
        img = np.array(Image.open(img_path).convert('RGB'))
        axes[row][col_start].imshow(img)
        axes[row][col_start].set_title(f'Imagem', fontsize=10)
    axes[row][col_start].axis('off')

    if mask_col:
        mask_path = find_file(str(entry[mask_col]))
        if mask_path:
            mask = np.array(Image.open(mask_path).convert('L'))
            axes[row][col_start+1].imshow(mask, cmap='gray')
            axes[row][col_start+1].set_title('Mascara', fontsize=10)
    axes[row][col_start+1].axis('off')

plt.suptitle('Amostras de Germinoma (Germ Cell Tumor)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Preparação dos Dados

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 8

class BrainTumorDataset(Dataset):
    def __init__(self, dataframe, img_col, mask_col, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_col = img_col
        self.mask_col = mask_col
        self.transform = transform

        # resolver caminhos
        self.img_paths = []
        self.mask_paths = []
        valid = []
        for idx in range(len(self.data)):
            ip = find_file(str(self.data[self.img_col].iloc[idx]))
            mp = find_file(str(self.data[self.mask_col].iloc[idx]))
            if ip and mp:
                self.img_paths.append(ip)
                self.mask_paths.append(mp)
                valid.append(idx)
        self.data = self.data.iloc[valid].reset_index(drop=True)
        print(f'  {len(self.img_paths)} pares imagem/mascara validos')

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.img_paths[idx]).convert('RGB'))
        mask = np.array(Image.open(self.mask_paths[idx]).convert('L'))
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug['image']
            mask = aug['mask']

        if isinstance(mask, torch.Tensor) and mask.dim() == 2:
            mask = mask.unsqueeze(0)
        elif isinstance(mask, np.ndarray) and mask.ndim == 2:
            mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask

In [ ]:
# Transforms
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=20, p=0.4),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Split 70/15/15
train_df, temp_df = train_test_split(df_germ, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f'Treino: {len(train_df)} | Validacao: {len(val_df)} | Teste: {len(test_df)}')

train_dataset = BrainTumorDataset(train_df, img_col, mask_col, train_transform)
val_dataset = BrainTumorDataset(val_df, img_col, mask_col, val_transform)
test_dataset = BrainTumorDataset(test_df, img_col, mask_col, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 4. Metodo da Literatura - U-Net com ResNet34

A U-Net (Ronneberger et al., 2015) eh a arquitetura mais usada na literatura para segmentacao de imagens medicas. Aqui usamos um encoder ResNet34 pre-treinado no ImageNet, que eh uma abordagem bastante comum nos notebooks do Kaggle para esse tipo de problema.

Referencia: notebooks disponibilizados na aba "Code" do dataset no Kaggle.

In [ ]:
# Funcoes de metricas
def dice_score(preds, targets, smooth=1e-6):
    preds_flat = preds.view(-1)
    targets_flat = targets.view(-1)
    intersection = (preds_flat * targets_flat).sum()
    return (2. * intersection + smooth) / (preds_flat.sum() + targets_flat.sum() + smooth)

def iou_score(preds, targets, smooth=1e-6):
    preds_flat = preds.view(-1)
    targets_flat = targets.view(-1)
    intersection = (preds_flat * targets_flat).sum()
    union = preds_flat.sum() + targets_flat.sum() - intersection
    return (intersection + smooth) / (union + smooth)

# Funcao de treino
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    total_dice = 0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        loss = criterion(outputs, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        total_loss += loss.item()
        total_dice += dice_score(preds, masks).item()
    return total_loss / len(loader), total_dice / len(loader)

# Funcao de avaliacao
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_dice, total_iou = 0, 0, 0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        loss = criterion(outputs, masks)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        total_loss += loss.item()
        total_dice += dice_score(preds, masks).item()
        total_iou += iou_score(preds, masks).item()
    n = len(loader)
    return total_loss/n, total_dice/n, total_iou/n

In [ ]:
# Criar modelo U-Net + ResNet34
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None
).to(device)

# Loss: Dice + BCE
dice_loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)
bce_loss_fn = nn.BCEWithLogitsLoss()
def criterion(pred, target):
    return 0.5 * dice_loss_fn(pred, target) + 0.5 * bce_loss_fn(pred, target)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

n_params = sum(p.numel() for p in model.parameters())
print(f'Modelo: U-Net + ResNet34')
print(f'Parametros: {n_params:,}')

In [ ]:
# Treinar
EPOCHS = 30
history = {'train_loss': [], 'train_dice': [], 'val_loss': [], 'val_dice': [], 'val_iou': []}
best_dice = 0

for epoch in range(EPOCHS):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice, val_iou = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_dice)

    history['train_loss'].append(train_loss)
    history['train_dice'].append(train_dice)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)
    history['val_iou'].append(val_iou)

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), '/content/best_model.pth')

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f} - '
              f'Train Dice: {train_dice:.4f} - Val Dice: {val_dice:.4f} - Val IoU: {val_iou:.4f}')

print(f'\nMelhor Val Dice: {best_dice:.4f}')

In [ ]:
# Avaliar no teste
model.load_state_dict(torch.load('/content/best_model.pth'))
test_loss, test_dice, test_iou = evaluate(model, test_loader, criterion, device)

print(f'Resultados no Teste:')
print(f'  Loss: {test_loss:.4f}')
print(f'  Dice: {test_dice:.4f}')
print(f'  IoU:  {test_iou:.4f}')

In [ ]:
# Curvas de treinamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history['train_loss'], label='Treino')
axes[0].plot(epochs_range, history['val_loss'], label='Validacao')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoca')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_dice'], label='Treino Dice')
axes[1].plot(epochs_range, history['val_dice'], label='Val Dice')
axes[1].plot(epochs_range, history['val_iou'], label='Val IoU', linestyle='--')
axes[1].set_title('Metricas')
axes[1].set_xlabel('Epoca')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('U-Net + ResNet34 - Curvas de Treinamento', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar predicoes
model.eval()
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
indices = random.sample(range(len(test_dataset)), min(4, len(test_dataset)))

for i, idx in enumerate(indices):
    image, mask = test_dataset[idx]
    with torch.no_grad():
        pred = torch.sigmoid(model(image.unsqueeze(0).to(device)))
    pred = (pred > 0.5).float().cpu().squeeze().numpy()

    img_np = image.numpy().transpose(1, 2, 0)
    img_np = (img_np * std + mean).clip(0, 1)
    mask_np = mask.squeeze().numpy()

    d = dice_score(torch.from_numpy(pred).unsqueeze(0), mask.squeeze().unsqueeze(0)).item()

    axes[i][0].imshow(img_np)
    axes[i][0].set_title('Imagem Original')
    axes[i][0].axis('off')

    axes[i][1].imshow(mask_np, cmap='gray')
    axes[i][1].set_title('Mascara Real')
    axes[i][1].axis('off')

    axes[i][2].imshow(pred, cmap='gray')
    axes[i][2].set_title(f'Predicao (Dice: {d:.3f})')
    axes[i][2].axis('off')

plt.suptitle('Predicoes do Modelo - U-Net + ResNet34', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Ideias para Metodos Proprios

Com base no metodo da literatura testado acima (U-Net + ResNet34), propomos as seguintes ideias de melhorias que podem ser exploradas em proximas etapas:

### 5.1 Otimizacao de Hiperparametros

- **Learning rate**: Testar valores como 5e-4 ou 1e-3 com CosineAnnealingLR no lugar do ReduceLROnPlateau, para um decaimento mais suave.
- **Loss function**: Trocar a Dice+BCE por **Tversky Loss** com alpha=0.7 e beta=0.3, que penaliza mais os falsos negativos (nao detectar o tumor eh pior do que um alarme falso).
- **Batch size**: Testar batch size 4 ou 16 para avaliar o impacto.

### 5.2 Arquitetura Mais Leve

- Substituir o encoder **ResNet34** por **MobileNetV2**, que tem muito menos parametros e treina mais rapido. A biblioteca `segmentation-models-pytorch` suporta essa troca facilmente:

```python
model_leve = smp.Unet(
    encoder_name='mobilenet_v2',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1
)
```

- Isso reduziria significativamente o tempo de treino e inferencia, sendo mais adequado para dispositivos com menos poder computacional.

### 5.3 Reducao do Numero de Imagens

- Treinar com apenas **50% das imagens** de treino e compensar com **data augmentation mais agressiva** (ElasticTransform, GridDistortion, CoarseDropout).
- Isso testaria a robustez do modelo e a eficacia das tecnicas de augmentation para datasets pequenos como o nosso (263 imagens de Germinoma).

### 5.4 Outras Ideias

- **Attention U-Net**: Adicionar mecanismos de atencao no decoder para focar melhor nas regioes do tumor.
- **Ensemble**: Combinar predicoes de modelos com encoders diferentes (ResNet34 + MobileNetV2) por media.
- **Pos-processamento**: Aplicar operacoes morfologicas (erosao/dilatacao) nas mascaras preditas para remover ruido.

## 6. Conclusao

Nesta etapa (ET-02), executamos o metodo da literatura (U-Net com encoder ResNet34 pre-treinado) aplicado a segmentacao de Germ Cell Tumors (Germinomas) em imagens de MRI cerebral.

O modelo foi treinado com 263 imagens do dataset, divididas em treino/validacao/teste (70/15/15%), e as metricas obtidas (Dice Score e IoU) mostram a viabilidade da abordagem para este tipo de tumor.

Tambem foram propostas ideias de melhorias para metodos proprios, incluindo otimizacao de hiperparametros, uso de arquiteturas mais leves (MobileNetV2), e reducao do volume de dados com augmentation.

---

**Referencias:**
- Ronneberger, O. et al. (2015). U-Net: Convolutional Networks for Biomedical Image Segmentation.
- He, K. et al. (2016). Deep Residual Learning for Image Recognition.
- Dataset: Feltrin, F. Brain Tumor 12K MRI Images w Masks, Meta and Bbox. Kaggle.